In [ ]:
#Instalar los requerimientos de Huggin Face
!pip install -U transformers

## Transcripción
Usando el modelo Whisper de OpenAI (https://huggingface.co/openai/whisper-large-v3-turbo) creamos una pipeline de Hugging Face dedicada a la transcripción.

Las pipelines son objetos que simplifican el uso de distintos modelos de Hugging Face. Existen varios tipos, diferenciados según la tarea a la que están orientados.

Cada tarea requiere distintos parámetros. Por ejemplo, el parámetro return_timestamps obtiene el tiempo (en segundos) en que empieza y termina cada frase transcrita. Este es un parámetro específico de la pipeline de reconocimiento de voz.

In [ ]:
#Crear modelo
from transformers import pipeline #La pipeline ya te prepara todo para que se pueda usar , es una manera de ejcutarlo , solo hay que especificar lo de abajo y ya

pipe = pipeline("automatic-speech-recognition", model="openai/whisper-large-v3-turbo")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
Device set to use cuda:0


In [ ]:
import os
import librosa # libreria para manejar audio
import json

def transcribe_audio_to_json(filename):
    cwd = os.getcwd()
    audio_path = os.path.join(cwd, filename)

    try:
        # Leer el audio usando librosa
        audio_data, sample_rate = librosa.load(audio_path, sr=16000)
        # En principio funciona tambien con formato video

        # Usamos la pipeline con el argumento de return_timestamps=True
        # así obtenemos el diálogo segmentado con cuándo empieza y acaba
        result = pipe(audio_data, return_timestamps=True)
        # Eliminamos la extensión del nombre del archivo (.mp3, .wav...)
        json_name = f"{os.path.splitext(filename)[0]}.json"
        json_dir = os.path.join(cwd, "json")
        if not os.path.isdir(json_dir):
            os.makedirs(json_dir)
        json_path = os.path.join(json_dir, json_name)

        try:
            #  Guardar en un archivo json
            with open(json_path, "w", encoding="utf-8") as archivo:
                archivo.write(json.dumps(result))
            print(f"\033[32mArchivo guardado: {json_path}\033[0m")
        except FileNotFoundError:
            # Si la carpeta "json" no existe
            print(f"Error: El directorio no existe {json_path}")
        except Exception as e:
            print(f"Error procesando audio: {e}")

    # Manejo de errores
    except FileNotFoundError:
        print(f"Error: No se encontró el archivo {audio_path}")
    except Exception as e:
        print(f"Error procesando audio: {e}")



In [ ]:
import json

def read_json(json_path):
    try:
        with open(json_path, "r", encoding="utf-8") as f:
            transcript = json.loads(f.read())
            return transcript
    except Exception as e:
        print(f"Error: {e}")


## Traducción
Para traducir, tenemos que cargar un modelo de traducción. Nosotros usamos los modelos de Helsinki, por la cantidad de idiomas disponibles.

Como hay modelos distintos según los idiomas entre los que se traduce, tenemos que detectar el idioma de la transcripción. Una vez tenemos lo tenemos, usamos el idioma que haya seleccionado el usuario para traducir la transcripción.

In [ ]:
# Necesita saber cuál es el idioma de la transcripción
# para poder cargar el modelo correspondiente
def detect_language(text):
  text_lang_detection_pipe = pipeline("text-classification", model="papluca/xlm-roberta-base-language-detection")
  res = text_lang_detection_pipe(text)
  return res[0]['label']

# Como no sabemos en qué orden está, probamos a cargar el modelo
# de las dos formas. Si no existe, lanza un error
def load_translation_pipeline(from_lang, to_lang):
  try:
    model = f"Helsinki-NLP/opus-mt-{from_lang}-{to_lang}"
    translation_pipeline = pipeline("translation", model=model)
  except Exception as e:
    try:
      model = f"Helsinki-NLP/opus-mt-{to_lang}-{from_lang}"
      translation_pipeline = pipeline("translation", model=model)
    except Exception as e:
      print(f"Error: {e}")
  return translation_pipeline

def translate(translation_pipeline, text):
  res = translation_pipeline(text)
  return res[0]['translation_text']

Utilizamos las funciones para traducir la transcripción por fragmentos (chunk a chunk):

In [ ]:
# Traducir
def translate_transcript(input_lang, transcript):
  from_lang = detect_language(transcript['text'])
  langs_dict = read_json("lang_codes.json")
  # Coge el código de lenguaje del json
  # si el usuario pone "spanish", el código es "es"
  to_lang = langs_dict[input_lang]

  # Cargamos la pipeline de traducción una sola vez
  # y la usamos para cada chunk
  translation_pipe = load_translation_pipeline(from_lang, to_lang)

  chunks = transcript['chunks']
  new_chunks = []
  for chunk in chunks:
    translated_text = translate(translation_pipe, chunk['text'])
    new_chunks.append({'timestamp': chunk['timestamp'], 'text': translated_text})

  # El texto que no usamos para el srt no es traducido
  return {'text': transcript['text'], 'chunks': new_chunks}

## Crear los subtítulos
El formato estándar para subtítulos es srt, un archivo de texto plano con esta estructura:

```
1
00:02:16,612 --> 00:02:19,376
Senator, we're making
our final approach into Coruscant.

2
00:02:19,482 --> 00:02:21,609
Very good, Lieutenant.

3
00:03:13,336 --> 00:03:15,167
We made it.
```

Tenemos que adaptar los fragmentos de texto con sus tiempos en segundos a esta estructura como describiremos en el código a continuación.

In [ ]:
# Interpretamos el diccionario con el resultado:
# descartamos la transcripción entera (text) y nos guardamos los chunks

# Chunks (fragmentos) es un array
# Cada chunk tiene:
# - timestamp, una tupla con el tiempo donde empieza y acaba la frase
# - text, la transcripción de la frase

# Hay que adaptar las timestamps al formato estándar:
# 00:00:00,000 horas:minutos:segundos,milisegundos
def format_timestamp(timestamp):
    hours = int(timestamp // 3600)
    minutes = int((timestamp % 3600) // 60)
    seconds = int(timestamp % 60)
    milliseconds = int(round((timestamp - int(timestamp)) * 1000))

    # :02d es un especificador de formato para un string:
    # 0 -> cubre el tamaño con 0
    # 2 -> el string siempre tamaño 2
    # d -> es un número decimal, un int
    return f"{hours:02d}:{minutes:02d}:{seconds:02d},{milliseconds:03d}"

# El formato de un srt línea a línea por cada fragmento es:
# El número del fragmento empezando por 1 (un contador)
# La marca de tiempo donde empieza y acaba 00:00:00,000 --> 00:00:00,000
# El texto (puede estar separado en varias líneas)
# Un salto de línea para separar entre fragmentos
def format_srt(chunks):
    srt = ""
    num = 1
    for chunk in chunks:
        start = format_timestamp(chunk["timestamp"][0])
        end = format_timestamp(chunk["timestamp"][1])
        text = chunk["text"]
        srt += f"{num}\n{start} --> {end}\n{text}\n\n"
        num += 1
    return srt


In [ ]:
# Guardamos el string que acabamos de formatear como srt
import os

def save_srt(srt, filename):
    dir = os.path.join(os.getcwd(), "subs")
    path = os.path.join(dir, filename + ".srt")
    try:
        if not os.path.isdir(dir):
            os.makedirs(dir)

        with open(path, "w") as f:
            f.write(srt)
    except FileNotFoundError:
        print(f"Error: El directorio no existe {path}")
    except Exception as e:
        print(f"Error guardando srt: {e}")

## Gradio

In [ ]:
import os
import gradio as gr
import shutil

def process_file(file, target_lang):
    # Guardamos el archivo subido (en Colab, usar copy en lugar de move)
    filename = os.path.basename(file.name if hasattr(file, "name") else file)
    input_path = os.path.join(os.getcwd(), filename)
    shutil.copy(file.name if hasattr(file, "name") else file, input_path)

    # Transcribir el audio
    transcribe_audio_to_json(filename)

    # ✅ Corrección aquí:
    json_dir = os.path.join(os.getcwd(), "json")
    json_name = f"{os.path.splitext(filename)[0]}.json"
    json_path = os.path.join(json_dir, json_name)

    transcript = read_json(json_path)

    # Si hay idioma destino, traducimos
    if target_lang and target_lang.strip():
        translated_transcript = translate_transcript(target_lang, transcript)
        chunks = translated_transcript["chunks"]
    else:
        chunks = transcript["chunks"]

    # Formateamos los subtítulos
    srt_content = format_srt(chunks)

    # Guardamos el archivo .srt
    srt_dir = os.path.join(os.getcwd(), "subs")
    os.makedirs(srt_dir, exist_ok=True)
    srt_path = os.path.join(srt_dir, f"{os.path.splitext(filename)[0]}.srt")

    with open(srt_path, "w", encoding="utf-8") as f:
        f.write(srt_content)

    return srt_content, srt_path  # Muestra y permite descargar


# INTERFAZ GRADIO
with gr.Blocks() as demo:
    gr.Markdown("## 🎬 Creador de subtítulos")

    gr.Markdown("### Inserta aquí tu película o audio (arrastra o haz click en el cuadro):")

    file_input = gr.File(label="Audio o Video", file_types=["audio", "video"])
    lang_input = gr.Textbox(label="Idioma destino (opcional)", placeholder="Ej: english, french, spanish...")
    srt_box = gr.Textbox(label="Subtítulos generados", lines=10)
    srt_file = gr.File(label="Descargar archivo .srt")
    run_button = gr.Button("Generar subtítulos")

    run_button.click(fn=process_file, inputs=[file_input, lang_input], outputs=[srt_box, srt_file])

demo.launch(share=True)








Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://c71ee010c93957398d.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
